University of Arizona

Author: Nestor Molina

# Movie/TV Show Classifier

In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_extraction import text
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score

from scipy.sparse import hstack

## Training

In [2]:
# Load training data
train = pd.read_csv('data/train.csv')

# Noticed some nan rows, remove them
train = train.dropna()

# Visually inspect 
train

,ID,TEXT,LABEL
0,327995652116863138,Carolyn Baker defines the niche of helping peo...,0
1,11848336500074516230,Just getting released from a six month drug re...,1
2,8453485550425672763,I was greatly dissappointed when I popped this...,0
3,13088897910749354342,This is a film that has garnered any interest ...,2
4,4199320520317837800,This is one of the more adorable episodes of t...,1
...,...,...,...
70312,4095283484534003747,This review owes its existence entirely to a r...,1
70313,15401682080725988675,Even if you could get past the idea that these...,2
70314,9279728677504091897,I grew up watching just about all of the Disne...,1
70315,16755456328208390817,It's difficult to precisely put into words the...,2


In [3]:
def process_ratings(document):
    '''
    Replace anything that looks like a ratio rating with special
    token
    '''
    rating_pattern = r'(\d+)/(10+)'

    def ratingToToken(match):
        num = int(match[1])
        den = int(match[2])

        if num == 0 or den == 0:
            return '<RATING-0>'

        return f'<RATING-{round(num / den * 5)}>'
    
    return re.sub(
        rating_pattern,
        ratingToToken,
        document
    )

def prepareData(data):
    '''
    Apply some preprocessing to the yet untouched
    data
    '''
    print('Preparing Data')

    X = data['TEXT']
    Y = data['LABEL'] if 'LABEL' in data.columns else None

    # Remove non ascii characters
    X = X.str.replace(r'[^\x00-\x7F]+', '', regex = True)

    # Separte contractions
    X = X.str.replace("'", '')

    X = X.apply(process_ratings)

    # Remove whitespace from ends
    # Add document start and end markers
    X = '<SOS> ' + X.str.strip() + ' <EOS>'

    return X, Y

In [4]:
def prepareCountVectorizer(X):
    '''
    Fit count/tfidf vectorizer
    Return transformed data
    '''
    print('Preparing Count Vectorizer')

    sentence_tags = r'<SOS>|<EOS>'
    rating_tags = r'<RATING-\d+>'
    
    # Create token pattern
    token_pattern = rf'(?u)(?:{sentence_tags})|(?:{rating_tags})|\b\w\w+\b'
    
    # count_vect = CountVectorizer(
    #     analyzer = 'word',
    #     token_pattern = token_pattern,
    #     ngram_range = (1, 2),
    #     lowercase = True,
    #     binary = True,
    # )

    # Exclude some words from stop word filtertting
    stop_words = set(text.ENGLISH_STOP_WORDS)
    important_words = {
        'not', 'no', 'never', 'neither', 'nor', 'none', 
        'but', 'however', 'yet', 
        'very', 'extremely', 'too', 'more', 'most', 'least',
        'against', 'cannot', 'couldnt', 'didnt', 'doesnt', 'dont', 'isnt'
    }
    custom_stop_words = list(stop_words - important_words)

    count_vect = TfidfVectorizer(
        analyzer = 'word',
        token_pattern = token_pattern,
        ngram_range = (1, 2),
        max_features = 50000, 
        min_df = 3, 
        max_df = 0.9, 
        stop_words = custom_stop_words,
        lowercase = True,
    )

    x_counts = count_vect.fit_transform(X)

    return x_counts, count_vect

In [5]:
def stringToDict(document):
    '''
    Extract some extra features
    '''
    # Remove sos/eos tokens
    doc = document[6:-6]
    return {
        'all_caps': doc.isupper(),
        'has_alpha': any(c.isalpha() for c in doc),
        'num_new_lines': doc.count('\n'),
        'len': np.log(len(doc.split()) + 1),
    }

def prepareDictVectorizer(X):
    '''
    Fit a dict vectorizer
    Transform data
    '''
    print('Preparing Dict Vectorizer')

    dict_vect = DictVectorizer()
    
    dict_list = X.map(stringToDict)
    
    x_dict = dict_vect.fit_transform(dict_list)

    return x_dict, dict_vect

In [6]:
def crossValidateModel(X, Y):
    '''
    Use cross validation to gauge performance
    '''
    print('Cross-Validating')

    model = LogisticRegression(
        class_weight = 'balanced',
        max_iter = 10000,
    )
    
    scores = cross_val_score(
        model,
        x_features,
        Y,
        cv = 5,
        scoring = 'f1_macro',
        n_jobs = -2,
    )

    print('Scores:', scores)
    print('Mean score:', scores.mean())

    return model

In [7]:
# Split data
X, Y = prepareData(train)

# Get couns
x_counts, count_vect = prepareCountVectorizer(X)

# Get other features
x_dict, dict_vect = prepareDictVectorizer(X)

# Combine all features
x_features = hstack([x_counts, x_dict])

# Run cross validation
model = crossValidateModel(x_features, Y)

Preparing Data
Preparing Count Vectorizer
Preparing Dict Vectorizer
Cross-Validating
Scores: [0.9171809  0.91979243 0.91747337 0.91998347 0.92000686]
Mean score: 0.9188874054176193


## Prediction

In [9]:
# Train model on all data
model.fit(x_features, Y)

LogisticRegression(class_weight='balanced', max_iter=10000)

In [10]:
# Get test data
test = pd.read_csv('data/test.csv')

# Preproccess
x_test, _ = prepareData(test)

# Get features
x_test_counts = count_vect.transform(x_test)
x_test_dict = dict_vect.transform(
        x_test.map(stringToDict)
    )

# Combine features
x_test_features = hstack([x_test_counts, x_test_dict])

# Predict
y_pred = model.predict(x_test_features)

Preparing Data


In [11]:
test['LABEL'] = y_pred

# Save to submit to kaggle
test[['ID', 'LABEL']].to_csv('submission.csv', index = False)